In [ ]:
#| default_exp icons

In [ ]:
#| export
from __future__ import annotations
import struct, subprocess, zlib
from fastcore.all import Path

In [ ]:
#| export
#: The sizes an `.icns` carries, as `(pixels, iconset name)`. Retina entries are the same pixel
#: count as the next size up, which is why `icon_16x16@2x` and `icon_32x32` are both 32.
ICNS_SIZES = ((16, 'icon_16x16'), (32, 'icon_16x16@2x'), (32, 'icon_32x32'),
              (64, 'icon_32x32@2x'), (128, 'icon_128x128'), (256, 'icon_128x128@2x'),
              (256, 'icon_256x256'), (512, 'icon_256x256@2x'), (512, 'icon_512x512'),
              (1024, 'icon_512x512@2x'))

#: What a Windows `.ico` holds. Above 256 the format stores a PNG, and nothing asks for one.
ICO_SIZES = (16, 24, 32, 48, 64, 128, 256)

In [ ]:
#| export
def _open(src):
    "The source image as RGBA, square, or the reason it cannot be used."
    from PIL import Image
    im = Image.open(src).convert('RGBA')
    if im.width != im.height:
        raise ValueError(f'{src} is {im.width}x{im.height}; an app icon has to be square')
    if im.width < 512:
        raise ValueError(f'{src} is {im.width}px; macOS draws icons at 1024 and will upscale this')
    return im

def _resized(im, size):
    from PIL import Image
    return im.resize((size, size), Image.LANCZOS)

In [ ]:
#| export
def iconset(src, out):
    "The `.iconset` folder `iconutil` turns into an `.icns`. Returns the folder."
    im, out = _open(src), Path(out)
    out.mkdir(parents=True, exist_ok=True)
    for size, name in ICNS_SIZES: _resized(im, size).save(out/f'{name}.png')
    return out

def icns(src, out, keep_iconset=False):
    """An `.icns` from one square image, through macOS's own `iconutil`.

    `iconutil` is macOS-only, so this raises everywhere else rather than writing something a
    bundle would accept and then draw wrongly.
    """
    out = Path(out)
    folder = out.with_suffix('.iconset')
    iconset(src, folder)
    try: subprocess.run(['iconutil', '-c', 'icns', str(folder), '-o', str(out)], check=True,
                        capture_output=True)
    except FileNotFoundError as e:
        raise RuntimeError('an .icns is built by `iconutil`, which only macOS has') from e
    finally:
        if not keep_iconset:
            for f in folder.glob('*.png'): f.unlink()
            folder.rmdir()
    return out

In [ ]:
#| export
def ico(src, out, sizes=ICO_SIZES):
    "A Windows `.ico` from one square image. Pillow writes every size into the one file."
    im, out = _open(src), Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    im.save(out, format='ICO', sizes=[(s, s) for s in sizes])
    return out

In [ ]:
#| export
def favicon(src, out, size=64):
    """A `.png` favicon for the app's own web UI, so the tab and the Dock carry one mark.

    Small: below about 64px a detailed mark stops carrying its line work, and a browser tab is
    16px of it.
    """
    out = Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    _resized(_open(src), size).save(out)
    return out

def icons_for(src, assets, name):
    "Every icon asset a bundle wants, written under `assets`. Skips the ones this platform cannot make."
    import sys
    assets, out = Path(assets), {}
    assets.mkdir(parents=True, exist_ok=True)
    out['favicon'] = str(favicon(src, assets/'favicon.png'))
    out['ico'] = str(ico(src, assets/f'{name}.ico'))
    if sys.platform == 'darwin': out['icns'] = str(icns(src, assets/f'{name}.icns'))
    return out